# Tải Himawari-9 B13 trong một tháng, crop Việt Nam và lưu NetCDF

Notebook dành cho Google Colab. Hàm chính nhận `"YYYY-MM"` hoặc một ngày `"YYYY-MM-DD"` rồi xử lý toàn bộ tháng chứa ngày đó.

- Dữ liệu nguồn: NOAA Open Data, không cần tài khoản/API key.
- Chu kỳ: 30 phút, tương ứng 48 timestamp/ngày.
- Chỉ tải B13 và các segment 03–05 phủ Việt Nam.
- Từng timestamp được tải vào thư mục tạm, crop/resample xong thì file `.bz2` và `.DAT` tự xóa.
- Mỗi ngày được lưu thành một file `.nc`; cách này giảm RAM và có thể chạy tiếp sau khi Colab bị ngắt.
- Một ngày chỉ được lưu khi đủ 48 timestamp. File ngày đã đủ sẽ tự động được bỏ qua.

> Ước lượng theo thử nghiệm hiện tại: dữ liệu cuối khoảng 1–1.5 GB/tháng. Lượng dữ liệu truyền qua mạng lớn hơn vì phải tải HSD gốc trước khi crop.


In [ ]:
%pip install -q satpy pyresample xarray netCDF4 requests


In [ ]:
from calendar import monthrange
from concurrent.futures import ThreadPoolExecutor
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from tempfile import TemporaryDirectory
import bz2
import gc
import os
import sys
import warnings

import numpy as np
import requests
import xarray as xr
from pyresample import create_area_def
from satpy import Scene

INTERVAL_MINUTES = 30
BAND = "B13"
SEGMENTS = ["03", "04", "05"]
VIETNAM_BBOX = (102.0, 8.0, 110.0, 24.5)
GRID_RESOLUTION_DEGREES = 0.02
NOAA_BASE_URL = "https://noaa-himawari9.s3.amazonaws.com"
HTTP_TIMEOUT = (10, 180)
HTTP_RETRIES = 3
USE_GOOGLE_DRIVE = True
DRIVE_OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Rainfall_Nowcasting/Himawari/compact"
)

if "google.colab" in sys.modules and USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    OUTPUT_DIR = DRIVE_OUTPUT_DIR
elif "google.colab" in sys.modules:
    OUTPUT_DIR = Path("/content/himawari/compact")
else:
    OUTPUT_DIR = Path("Himawari/data/himawari/compact")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings(
    "ignore",
    message="invalid value encountered in log",
    category=RuntimeWarning,
)

if 24 * 60 % INTERVAL_MINUTES != 0:
    raise ValueError("INTERVAL_MINUTES phải chia hết 1440 phút.")
EXPECTED_SLOTS_PER_DAY = 24 * 60 // INTERVAL_MINUTES
vietnam_area = create_area_def(
    "vietnam_latlon",
    {"proj": "longlat", "datum": "WGS84"},
    area_extent=VIETNAM_BBOX,
    resolution=(GRID_RESOLUTION_DEGREES, GRID_RESOLUTION_DEGREES),
    units="degrees",
)
print("Output:", OUTPUT_DIR.resolve())
print(f"Số timestamp yêu cầu mỗi ngày: {EXPECTED_SLOTS_PER_DAY}")


In [ ]:
def parse_month(value: str | date | datetime) -> tuple[int, int]:
    """Lấy năm và tháng từ YYYY-MM, YYYY-MM-DD, YYYYMM hoặc YYYYMMDD."""
    if isinstance(value, (date, datetime)):
        return value.year, value.month

    text = str(value).strip()
    for format_string in ("%Y-%m", "%Y-%m-%d", "%Y%m", "%Y%m%d"):
        try:
            parsed = datetime.strptime(text, format_string)
            return parsed.year, parsed.month
        except ValueError:
            continue
    raise ValueError(
        "Ngày không hợp lệ. Dùng YYYY-MM hoặc YYYY-MM-DD, ví dụ 2026-08."
    )


def get_month_dates(value: str | date | datetime) -> list[date]:
    """Trả về các ngày hoàn chỉnh trong tháng theo giờ UTC."""
    year, month = parse_month(value)
    first_day = date(year, month, 1)
    last_day = date(year, month, monthrange(year, month)[1])
    today_utc = datetime.now(timezone.utc).date()

    if first_day > today_utc:
        raise ValueError("Không thể tải một tháng nằm trong tương lai.")
    if (year, month) == (today_utc.year, today_utc.month):
        last_day = min(last_day, today_utc - timedelta(days=1))
    if last_day < first_day:
        raise ValueError("Tháng hiện tại chưa có ngày UTC hoàn chỉnh để tải.")

    number_of_days = (last_day - first_day).days + 1
    return [first_day + timedelta(days=offset) for offset in range(number_of_days)]


def daily_slots(day: date) -> list[datetime]:
    start = datetime(day.year, day.month, day.day)
    return [
        start + timedelta(minutes=minute)
        for minute in range(0, 24 * 60, INTERVAL_MINUTES)
    ]


def daily_output_path(day: date) -> Path:
    return OUTPUT_DIR / (
        f"H09_{day:%Y%m%d}_{BAND}_Vietnam_{INTERVAL_MINUTES}min.nc"
    )


def is_complete_daily_file(path: Path) -> bool:
    if not path.exists():
        return False
    try:
        with xr.open_dataset(path) as dataset:
            return (
                BAND in dataset
                and dataset.sizes.get("time") == EXPECTED_SLOTS_PER_DAY
                and dataset.sizes.get("latitude", 0) > 0
                and dataset.sizes.get("longitude", 0) > 0
            )
    except (OSError, ValueError):
        return False


In [ ]:
def make_filename(slot: datetime, segment: str) -> str:
    return (
        f"HS_H09_{slot:%Y%m%d}_{slot:%H%M}_{BAND}_"
        f"FLDK_R20_S{segment}10.DAT"
    )


def make_url(slot: datetime, filename: str) -> str:
    key = f"AHI-L1b-FLDK/{slot:%Y/%m/%d/%H%M}/{filename}.bz2"
    return f"{NOAA_BASE_URL}/{key}"


def download_file(url: str, destination: Path) -> None:
    temporary = destination.with_suffix(destination.suffix + ".part")
    for attempt in range(1, HTTP_RETRIES + 1):
        try:
            with requests.get(url, stream=True, timeout=HTTP_TIMEOUT) as response:
                response.raise_for_status()
                with temporary.open("wb") as output_file:
                    for chunk in response.iter_content(1024 * 1024):
                        if chunk:
                            output_file.write(chunk)
            os.replace(temporary, destination)
            return
        except requests.RequestException:
            temporary.unlink(missing_ok=True)
            if attempt == HTTP_RETRIES:
                raise


def decompress_file(source: Path, destination: Path) -> None:
    with bz2.open(source, "rb") as input_file:
        with destination.open("wb") as output_file:
            while chunk := input_file.read(1024 * 1024):
                output_file.write(chunk)


def prepare_segment(slot: datetime, segment: str, work_dir: Path) -> Path:
    filename = make_filename(slot, segment)
    compressed_path = work_dir / f"{filename}.bz2"
    dat_path = work_dir / filename
    download_file(make_url(slot, filename), compressed_path)
    decompress_file(compressed_path, dat_path)
    return dat_path


In [ ]:
def process_slot(slot: datetime) -> xr.DataArray:
    """Tải, giải nén, crop và trả về đúng một khung thời gian B13."""
    with TemporaryDirectory(prefix="himawari_") as temporary_dir:
        work_dir = Path(temporary_dir)
        with ThreadPoolExecutor(max_workers=len(SEGMENTS)) as executor:
            dat_files = list(
                executor.map(
                    lambda segment: prepare_segment(slot, segment, work_dir),
                    SEGMENTS,
                )
            )

        scene = Scene(reader="ahi_hsd", filenames=[str(path) for path in dat_files])
        scene.load([BAND], calibration="brightness_temperature")
        cropped_scene = scene.crop(ll_bbox=VIETNAM_BBOX)
        resample_scene = Scene()
        resample_scene[BAND] = cropped_scene[BAND]
        latlon_scene = resample_scene.resample(
            vietnam_area,
            datasets=[BAND],
            resampler="nearest",
            unload=False,
        )
        data = latlon_scene[BAND].rename({"y": "latitude", "x": "longitude"})
        data = data.astype(np.float32).compute().sortby("latitude")

    timestamp = np.datetime64(slot.strftime("%Y-%m-%dT%H:%M"))
    data = data.expand_dims(time=[timestamp]).copy(deep=True)
    data.name = BAND
    data.attrs = {
        "long_name": "Himawari-9 AHI B13 brightness temperature",
        "units": "K",
        "source": "NOAA Open Data, AHI Level-1b HSD",
        "spatial_resolution_degrees": GRID_RESOLUTION_DEGREES,
    }
    data.latitude.attrs = {"standard_name": "latitude", "units": "degrees_north"}
    data.longitude.attrs = {"standard_name": "longitude", "units": "degrees_east"}
    return data


In [ ]:
def save_daily_file(frames: list[xr.DataArray], day: date, output_path: Path) -> None:
    daily_data = xr.concat(frames, dim="time").sortby("time")
    dataset = daily_data.to_dataset(name=BAND)
    dataset = dataset.drop_vars("crs", errors="ignore")
    dataset.time.attrs = {"standard_name": "time", "timezone": "UTC"}
    dataset.attrs = {
        "title": "Himawari-9 B13 cropped to Vietnam bounding box",
        "date_utc": day.isoformat(),
        "bbox": "102E, 8N, 110E, 24.5N",
        "source_format": "Himawari Standard Data (HSD)",
        "interval_minutes": INTERVAL_MINUTES,
    }
    chunk_shape = (
        1,
        min(256, dataset.sizes["latitude"]),
        min(256, dataset.sizes["longitude"]),
    )
    encoding = {
        BAND: {
            "dtype": "float32",
            "zlib": True,
            "complevel": 4,
            "shuffle": True,
            "chunksizes": chunk_shape,
        }
    }
    temporary_output = output_path.with_suffix(".nc.part")
    temporary_output.unlink(missing_ok=True)
    try:
        dataset.to_netcdf(
            temporary_output,
            engine="netcdf4",
            format="NETCDF4",
            encoding=encoding,
        )
        os.replace(temporary_output, output_path)
    finally:
        temporary_output.unlink(missing_ok=True)
        dataset.close()


def download_day(day: date, overwrite: bool = False) -> dict:
    output_path = daily_output_path(day)
    if not overwrite and is_complete_daily_file(output_path):
        size_mb = output_path.stat().st_size / 1024**2
        print(f"SKIP {day}: đã đủ {EXPECTED_SLOTS_PER_DAY} timestamp ({size_mb:.1f} MB)")
        return {"date": day, "status": "skipped", "path": output_path}

    slots = daily_slots(day)
    frames = []
    failures = []
    for index, slot in enumerate(slots, start=1):
        try:
            frames.append(process_slot(slot))
            print(f"  [{index:02d}/{len(slots)}] OK {slot:%H:%M} UTC")
        except Exception as error:
            failures.append((slot, str(error)))
            print(f"  [{index:02d}/{len(slots)}] LỖI {slot:%H:%M}: {error}")

    if failures:
        del frames
        gc.collect()
        failed_times = ", ".join(slot.strftime("%H:%M") for slot, _ in failures)
        raise RuntimeError(
            f"{day} thiếu {len(failures)} timestamp ({failed_times}); chưa ghi file ngày."
        )

    save_daily_file(frames, day, output_path)
    del frames
    gc.collect()
    size_mb = output_path.stat().st_size / 1024**2
    print(f"DONE {day}: {output_path.name} ({size_mb:.1f} MB)")
    return {"date": day, "status": "downloaded", "path": output_path}


In [ ]:
def download_month(
    date_value: str | date | datetime,
    overwrite: bool = False,
    stop_on_error: bool = False,
) -> list[dict]:
    """Tải toàn bộ các ngày hoàn chỉnh của tháng chứa date_value."""
    days = get_month_dates(date_value)
    year, month = parse_month(date_value)
    print(f"\nBắt đầu tháng {year:04d}-{month:02d}: {len(days)} ngày")
    print(f"Mỗi ngày: {EXPECTED_SLOTS_PER_DAY} timestamp, output: 1 file NetCDF\n")

    results = []
    for day_index, day in enumerate(days, start=1):
        print(f"=== NGÀY {day_index}/{len(days)}: {day} ===")
        try:
            results.append(download_day(day, overwrite=overwrite))
        except Exception as error:
            print(f"FAILED {day}: {error}")
            results.append(
                {"date": day, "status": "failed", "error": str(error)}
            )
            if stop_on_error:
                raise

    downloaded = sum(item["status"] == "downloaded" for item in results)
    skipped = sum(item["status"] == "skipped" for item in results)
    failed = sum(item["status"] == "failed" for item in results)
    print("\n=== TỔNG KẾT ===")
    print(f"Đã tải: {downloaded} ngày | Đã có sẵn: {skipped} | Lỗi: {failed}")
    if failed:
        print("Chạy lại cùng lệnh để tải lại các ngày lỗi; ngày hoàn chỉnh sẽ được bỏ qua.")
    return results


## Chạy tải một tháng

Đổi giá trị trong lệnh dưới đây. Cả `"2026-08"` và `"2026-08-15"` đều tải toàn bộ tháng 08/2026. Nếu chọn tháng UTC hiện tại, notebook chỉ tải đến hết ngày hôm qua vì ngày hôm nay chưa đủ 48 timestamp.


In [ ]:
results = download_month("2026-08")


In [ ]:
# Kiểm tra các file đã lưu trong tháng vừa chạy.
for item in results:
    if item["status"] in {"downloaded", "skipped"}:
        path = item["path"]
        with xr.open_dataset(path) as dataset:
            print(
                path.name,
                dict(dataset.sizes),
                f"{path.stat().st_size / 1024**2:.1f} MB",
            )
